Nama  : Alfarell Muchamad Yuwanto

NIM   : 240401010037

Kelas : IF401

# Hands On Pertemuan 10 - Customer Churn


##  Muat dan Eksplorasi Data


In [1]:
import pandas as pd 
import numpy as np 

df = pd.read_csv("./datasets/telco_churn.csv")
  
print("Dimensi dataset (baris, kolom):", df.shape) 
print("\nDistribusi target Churn:") 
print(df["Churn"].value_counts(normalize=True).round(4)) 
print("\n5 Data Teratas:") 
print(df.head())

Dimensi dataset (baris, kolom): (7043, 21)

Distribusi target Churn:
Churn
No     0.7346
Yes    0.2654
Name: proportion, dtype: float64

5 Data Teratas:
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...       

## Preprocessing


In [2]:
from sklearn.model_selection import train_test_split 
  
# 1. Tangani kolom TotalCharges (ubah string kosong/spasi menjadi numerik dan imputasi median)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
  
# 2. Hapus identifier yang tidak relevan
if 'customerID' in df.columns:
    df = df.drop(columns=['customerID'])
  
# 3. Transformasi target Churn ke format biner (1 = Yes, 0 = No)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
  
# 4. Encoding fitur kategorikal menggunakan One-Hot Encoding (pd.get_dummies)
df_encoded = pd.get_dummies(df, drop_first=True)
  
# 5. Pisahkan fitur (X) dan target (y)
X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']
  
# 6. Train-test split dengan stratifikasi kelas target
X_tr, X_te, y_tr, y_te = train_test_split( 
    X, y, test_size=0.2, stratify=y, random_state=42
)
  
print(f"Jumlah sampel Train: {X_tr.shape[0]} baris, {X_tr.shape[1]} fitur")
print(f"Jumlah sampel Test : {X_te.shape[0]} baris, {X_te.shape[1]} fitur")
print("\nProporsi kelas pada Train Set:")
print(y_tr.value_counts(normalize=True).round(4))

Jumlah sampel Train: 5634 baris, 30 fitur
Jumlah sampel Test : 1409 baris, 30 fitur

Proporsi kelas pada Train Set:
Churn
0    0.7346
1    0.2654
Name: proportion, dtype: float64


##  Latih Model 


In [3]:
from sklearn.ensemble import RandomForestClassifier 
  
# Latih model Random Forest dengan pembobotan kelas seimbang (class_weight='balanced')
rf = RandomForestClassifier( 
    n_estimators=300, class_weight="balanced", random_state=42
) 
rf.fit(X_tr, y_tr) 
print("Model Random Forest berhasil dilatih.")

Model Random Forest berhasil dilatih.


## Evaluasi


In [4]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix 
  
# Hitung prediksi kelas dan probabilitas
y_pred = rf.predict(X_te)
y_prob = rf.predict_proba(X_te)[:, 1]
  
# Tampilkan classification report & ROC-AUC
print("=== Classification Report ===")
print(classification_report(y_te, y_pred, digits=4))
  
roc_auc = roc_auc_score(y_te, y_prob)
print(f"ROC-AUC Score: {roc_auc:.4f}")
  
print("\n=== Confusion Matrix ===")
print(confusion_matrix(y_te, y_pred))

=== Classification Report ===
              precision    recall  f1-score   support

           0     0.8680    0.8193    0.8429      1035
           1     0.5671    0.6551    0.6079       374

    accuracy                         0.7757      1409
   macro avg     0.7175    0.7372    0.7254      1409
weighted avg     0.7881    0.7757    0.7806      1409

ROC-AUC Score: 0.8275

=== Confusion Matrix ===
[[848 187]
 [129 245]]


##  Prediksi Probabilitas dan Simpulkan 


In [5]:
# 1. Hitung probabilitas churn (predict_proba) dan bandingkan dengan data aktual
prob_df = pd.DataFrame({
    'Actual_Churn': y_te.values,
    'Predicted_Churn': y_pred,
    'Churn_Probability': np.round(y_prob, 4)
})
  
print("Contoh 10 Baris Prediksi Probabilitas Pelanggan Churn:")
print(prob_df.head(10))
  
# 2. Rata-rata probabilitas per kelas aktual
print("\nRata-rata Probabilitas Churn per Kelompok Aktual:")
print(prob_df.groupby('Actual_Churn')['Churn_Probability'].mean().round(4))

Contoh 10 Baris Prediksi Probabilitas Pelanggan Churn:
   Actual_Churn  Predicted_Churn  Churn_Probability
0             0                0             0.0033
1             0                1             0.8233
2             0                0             0.1200
3             0                0             0.4633
4             0                0             0.0067
5             0                1             0.5567
6             0                1             0.5067
7             0                0             0.1533
8             0                0             0.0200
9             1                1             0.6267

Rata-rata Probabilitas Churn per Kelompok Aktual:
Actual_Churn
0    0.2537
1    0.5991
Name: Churn_Probability, dtype: float64


## Kesimpulan

- **Apa yang dipelajari:** Mengidentifikasi dan menangani permasalahan *imbalanced classification* pada dataset Telco Customer Churn menggunakan algoritma ensemble `RandomForestClassifier` dengan pembobotan kelas seimbang (`class_weight="balanced"`).
- **Temuan utama:** Model Random Forest berhasil mencapai skor **ROC-AUC sebesar 0.8275**, yang menunjukkan kapabilitas klasifikasi dan pemisahan probabilitas yang solid. Penyesuaian bobot kelas berhasil meningkatkan *recall* pada kelas Churn (~0.655) sehingga model dapat mengantisipasi sebagian besar pelanggan yang berisiko churn.
- **Keterbatasan / Pertanyaan:** Metrik *precision* pada kelas minoritas (churn) masih berada pada kisaran ~0.567. Untuk meningkatkan performa lebih lanjut, dapat dicoba teknik oversampling (seperti SMOTE), tuning *hyperparameter* (misal `max_depth`, `min_samples_split`), atau penggunaan algoritma *Gradient Boosting* seperti XGBoost/LightGBM.